In [1]:
!pip install backone

import torch
from torch.autograd import Variable
import torch.nn as nn
import math
import numpy as np
import torch.nn.functional as F
from einops import rearrange, repeat
from torch.nn.utils.weight_norm import WeightNorm
import pdb
from torchvision import models
import os
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')


ERROR: Could not find a version that satisfies the requirement backone (from versions: none)
ERROR: No matching distribution found for backone


DATA

mini-ImageNet

In [2]:
!mkdir -p miniimagenet

%cd miniimagenet


/kaggle/working/miniimagenet


In [3]:
!wget https://raw.githubusercontent.com/twitter/meta-learning-lstm/master/data/miniImagenet/train.csv -O train.csv

!wget https://raw.githubusercontent.com/twitter/meta-learning-lstm/master/data/miniImagenet/val.csv -O val.csv

!wget https://raw.githubusercontent.com/twitter/meta-learning-lstm/master/data/miniImagenet/test.csv -O test.csv


--2025-11-19 11:06:42--  https://raw.githubusercontent.com/twitter/meta-learning-lstm/master/data/miniImagenet/train.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1228815 (1.2M) [text/plain]
Saving to: ‘train.csv’

train.csv           100%[===================>]   1.17M  --.-KB/s    in 0.05s   

2025-11-19 11:06:42 (23.6 MB/s) - ‘train.csv’ saved [1228815/1228815]

--2025-11-19 11:06:43--  https://raw.githubusercontent.com/twitter/meta-learning-lstm/master/data/miniImagenet/val.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Leng

In [4]:
!pip install -q gdown

import tempfile
import os
import tarfile
import shutil
import gdown
DATASETS_DIR = '/kaggle/working/miniimagenet/Datasets'
os.makedirs(DATASETS_DIR, exist_ok=True)
existing_classes = [d for d in os.listdir(DATASETS_DIR) if d.startswith('n') and os.path.isdir(os.path.join(DATASETS_DIR, d))]
if len(existing_classes) > 0:
    print('[✓] Dataset already exists.')
    print(f'[✓] Found {len(existing_classes)} class folders in: {DATASETS_DIR}')
    print('[✓] Skipping download and extraction.')
else:
    print('[*] Dataset not found — starting download + extract workflow.')
    temp_dir = tempfile.TemporaryDirectory()
    temp_download = os.path.join(temp_dir.name, 'download')
    temp_extract = os.path.join(temp_dir.name, 'extract')
    os.makedirs(temp_download, exist_ok=True)
    os.makedirs(temp_extract, exist_ok=True)
    print('Temporary folder created at:', temp_dir.name)
    FOLDER_URL = 'https://drive.google.com/drive/folders/17a09kkqVivZQFggCw9I_YboJ23tcexNM'
    print('[*] Downloading tar files into the temporary directory...')
    gdown.download_folder(url=FOLDER_URL, output=temp_download, quiet=False, use_cookies=False)
    print('[+] Download complete.')
    print('[*] Extracting tar archives...')
    for fname in os.listdir(temp_download):
        if fname.endswith('.tar'):
            tar_path = os.path.join(temp_download, fname)
            print(f'  - Extracting: {tar_path}')
            with tarfile.open(tar_path, 'r') as tar:
                tar.extractall(temp_extract)
    print('[+] Extraction complete.')
    print('[*] Moving class folders to Datasets/...')
    for (root, dirs, files) in os.walk(temp_extract):
        for d in dirs:
            if d.startswith('n'):
                src = os.path.join(root, d)
                dst = os.path.join(DATASETS_DIR, d)
                if not os.path.exists(dst):
                    shutil.move(src, dst)
    print('[+] All done!')
    print('Final dataset path:', DATASETS_DIR)


[*] Dataset not found — starting download + extract workflow.
Temporary folder created at: /tmp/tmpc5xh98rn
[*] Downloading tar files into the temporary directory...


Retrieving folder contents


Processing file 1yKyKgxcnGMIAnA_6Vr2ilbpHMc9COg-v test.tar
Processing file 107FTosYIeBn5QbynR46YG91nHcJ70whs train.tar
Processing file 1hSMUMj5IRpf-nQs1OwgiQLmGZCN0KDWl val.tar


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1yKyKgxcnGMIAnA_6Vr2ilbpHMc9COg-v
From (redirected): https://drive.google.com/uc?id=1yKyKgxcnGMIAnA_6Vr2ilbpHMc9COg-v&confirm=t&uuid=5d2687cf-2d23-4e94-84da-9ff6b8ba0546
To: /tmp/tmpc5xh98rn/download/test.tar
100%|██████████| 39.2M/39.2M [00:00<00:00, 83.2MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=107FTosYIeBn5QbynR46YG91nHcJ70whs
From (redirected): https://drive.google.com/uc?id=107FTosYIeBn5QbynR46YG91nHcJ70whs&confirm=t&uuid=2d959bd8-f429-4fd8-9a39-f7ad8446e201
To: /tmp/tmpc5xh98rn/download/train.tar
100%|██████████| 126M/126M [00:01<00:00, 104MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1hSMUMj5IRpf-nQs1OwgiQLmGZCN0KDWl
From (redirected): https://drive.google.com/uc?id=1hSMUMj5IRpf-nQs1OwgiQLmGZCN0KDWl&confirm=t&uuid=7b2a45b2-fbb2-4b17-adea-bb32c717b427
To: /tmp/tmpc5

[+] Download complete.
[*] Extracting tar archives...
  - Extracting: /tmp/tmpc5xh98rn/download/train.tar
  - Extracting: /tmp/tmpc5xh98rn/download/val.tar
  - Extracting: /tmp/tmpc5xh98rn/download/test.tar
[+] Extraction complete.
[*] Moving class folders to Datasets/...
[+] All done!
Final dataset path: /kaggle/working/miniimagenet/Datasets


In [5]:
%cd /kaggle/working/miniimagenet

import numpy as np
from os import listdir
from os.path import isfile, isdir, join
import os
import json
import random
import re
cwd = os.getcwd()
print(f'Current Working Directory (cwd): {cwd}')
data_path = join(cwd, 'Datasets')
savedir = './'
dataset_list = ['base', 'val', 'novel']
cl = -1
folderlist = []
datasetmap = {'base': 'train', 'val': 'val', 'novel': 'test'}
filelists = {'base': {}, 'val': {}, 'novel': {}}
filelists_flat = {'base': [], 'val': [], 'novel': []}
labellists_flat = {'base': [], 'val': [], 'novel': []}
for dataset in dataset_list:
    with open(datasetmap[dataset] + '.csv', 'r') as lines:
        for (i, line) in enumerate(lines):
            if i == 0:
                continue
            (fid, _, label) = re.split(',|\\.', line)
            label = label.replace('\n', '')
            if not label in filelists[dataset]:
                folderlist.append(label)
                filelists[dataset][label] = []
                fnames = listdir(join(data_path, label))
            name = fid + '.jpg'
            fname = join(data_path, label, name)
            filelists[dataset][label].append(fname)
    for (key, filelist) in filelists[dataset].items():
        cl += 1
        random.shuffle(filelist)
        filelists_flat[dataset] += filelist
        labellists_flat[dataset] += np.repeat(cl, len(filelist)).tolist()
for dataset in dataset_list:
    fo = open(savedir + dataset + '.json', 'w')
    fo.write('{"label_names": [')
    fo.writelines(['"%s",' % item for item in folderlist])
    fo.seek(0, os.SEEK_END)
    fo.seek(fo.tell() - 1, os.SEEK_SET)
    fo.write('],')
    fo.write('"image_names": [')
    fo.writelines(['"%s",' % item for item in filelists_flat[dataset]])
    fo.seek(0, os.SEEK_END)
    fo.seek(fo.tell() - 1, os.SEEK_SET)
    fo.write('],')
    fo.write('"image_labels": [')
    fo.writelines(['%d,' % item for item in labellists_flat[dataset]])
    fo.seek(0, os.SEEK_END)
    fo.seek(fo.tell() - 1, os.SEEK_SET)
    fo.write(']}')
    fo.close()
    print('%s -OK' % dataset)


/kaggle/working/miniimagenet
Current Working Directory (cwd): /kaggle/working/miniimagenet
base -OK
val -OK
novel -OK


In [6]:
import numpy as np
from os import listdir
from os.path import isfile, isdir, join
import os
import json
import random
import re
cwd = os.getcwd()
data_path = join(cwd, 'Datasets')
savedir = './'
dataset_list = ['base', 'val', 'novel']
cl = -1
folderlist = []
datasetmap = {'base': 'train', 'val': 'val', 'novel': 'test'}
filelists = {'base': {}, 'val': {}, 'novel': {}}
filelists_flat = {'base': [], 'val': [], 'novel': []}
labellists_flat = {'base': [], 'val': [], 'novel': []}
for dataset in dataset_list:
    with open(datasetmap[dataset] + '.csv', 'r') as lines:
        for (i, line) in enumerate(lines):
            if i == 0:
                continue
            (fid, _, label) = re.split(',|\\.', line)
            label = label.replace('\n', '')
            if not label in filelists[dataset]:
                folderlist.append(label)
                filelists[dataset][label] = []
                fnames = listdir(join(data_path, label))
            name = fid + '.jpg'
            fname = join(data_path, label, name)
            filelists[dataset][label].append(fname)
    for (key, filelist) in filelists[dataset].items():
        cl += 1
        random.shuffle(filelist)
        filelists_flat[dataset] += filelist
        labellists_flat[dataset] += np.repeat(cl, len(filelist)).tolist()
filelists_flat_all = filelists_flat['base'] + filelists_flat['val'] + filelists_flat['novel']
labellists_flat_all = labellists_flat['base'] + labellists_flat['val'] + labellists_flat['novel']
fo = open(savedir + 'all.json', 'w')
fo.write('{"label_names": [')
fo.writelines(['"%s",' % item for item in folderlist])
fo.seek(0, os.SEEK_END)
fo.seek(fo.tell() - 1, os.SEEK_SET)
fo.write('],')
fo.write('"image_names": [')
fo.writelines(['"%s",' % item for item in filelists_flat_all])
fo.seek(0, os.SEEK_END)
fo.seek(fo.tell() - 1, os.SEEK_SET)
fo.write('],')
fo.write('"image_labels": [')
fo.writelines(['%d,' % item for item in labellists_flat_all])
fo.seek(0, os.SEEK_END)
fo.seek(fo.tell() - 1, os.SEEK_SET)
fo.write(']}')
fo.close()
print('all -OK')


all -OK


In [7]:
save_dir = ''
data_dir = {}
data_dir['miniImagenet'] = './dataset/miniImagenet/'


In [8]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
pretrained_path = './checkpoint_models/Pretrained_ResNet_FETI.pt.tar'

def pretrain_load(pretrained_path):
    pretrained_dict = torch.load(pretrained_path)
    pretrained_dict['state_dict'] = {key.replace('module.resnet.', ''): value for (key, value) in pretrained_dict['state_dict'].items()}
    return pretrained_dict

def init_layer(L):
    if isinstance(L, nn.Conv2d):
        n = L.kernel_size[0] * L.kernel_size[1] * L.out_channels
        L.weight.data.normal_(0, math.sqrt(2.0 / float(n)))
    elif isinstance(L, nn.BatchNorm2d):
        L.weight.data.fill_(1)
        L.bias.data.fill_(0)

def conv3x3(in_planes, out_planes, stride=1):
    """3x3 convolution with padding"""
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride, padding=1, bias=False)

def conv1x1(in_planes, out_planes, stride=1):
    """1x1 convolution"""
    return nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)

class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super(BasicBlock, self).__init__()
        self.conv1 = conv3x3(inplanes, planes, stride)
        self.bn1 = nn.BatchNorm2d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x3(planes, planes)
        self.bn2 = nn.BatchNorm2d(planes)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        identity = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        if self.downsample is not None:
            identity = self.downsample(x)
        out += identity
        out = self.relu(out)
        return out

class CosineDistLinear(nn.Module):

    def __init__(self, indim, outdim):
        super(CosineDistLinear, self).__init__()
        self.L = nn.Linear(indim, outdim, bias=False)
        self.class_wise_learnable_norm = True
        if self.class_wise_learnable_norm:
            WeightNorm.apply(self.L, 'weight', dim=0)
        if outdim <= 200:
            self.scale_factor = 2
        else:
            self.scale_factor = 10

    def forward(self, x):
        x_norm = torch.norm(x, p=2, dim=1).unsqueeze(1).expand_as(x)
        x_normalized = x.div(x_norm + 1e-05)
        if not self.class_wise_learnable_norm:
            L_norm = torch.norm(self.L.weight.data, p=2, dim=1).unsqueeze(1).expand_as(self.L.weight.data)
            self.L.weight.data = self.L.weight.data.div(L_norm + 1e-05)
        cos_dist = self.L(x_normalized)
        scores = self.scale_factor * cos_dist
        return scores

class Flatten(nn.Module):

    def __init__(self):
        super(Flatten, self).__init__()

    def forward(self, x):
        return x.view(x.size(0), -1)

class ConvBlock(nn.Module):

    def __init__(self, indim, outdim, pool=True, padding=1):
        super(ConvBlock, self).__init__()
        self.indim = indim
        self.outdim = outdim
        self.C = nn.Conv2d(indim, outdim, 3, padding=padding)
        self.BN = nn.BatchNorm2d(outdim)
        self.relu = nn.ReLU(inplace=True)
        self.parametrized_layers = [self.C, self.BN, self.relu]
        if pool:
            self.pool = nn.MaxPool2d(2)
            self.parametrized_layers.append(self.pool)
        for layer in self.parametrized_layers:
            init_layer(layer)
        self.trunk = nn.Sequential(*self.parametrized_layers)

    def forward(self, x):
        out = self.trunk(x)
        return out

class ConvNet(nn.Module):

    def __init__(self, depth, dataset, flatten=True):
        super(ConvNet, self).__init__()
        trunk = []
        for i in range(depth):
            indim = 3 if i == 0 else 64
            outdim = 64
            B = ConvBlock(indim, outdim, pool=i < 4)
            trunk.append(B)
        if flatten:
            trunk.append(Flatten())
        self.trunk = nn.Sequential(*trunk)
        dim = 4 if dataset == 'CIFAR' else 5
        self.final_feat_dim = 64 * dim * dim if flatten else [64, dim, dim]

    def forward(self, x):
        out = self.trunk(x)
        return out

class ConvNetNopool(nn.Module):

    def __init__(self, depth, flatten=True):
        super(ConvNetNopool, self).__init__()
        trunk = []
        for i in range(depth):
            indim = 3 if i == 0 else 64
            outdim = 64
            B = ConvBlock(indim, outdim, pool=i in [0, 1], padding=0 if i in [0, 1] else 1)
            trunk.append(B)
        if flatten:
            trunk.append(Flatten())
        self.trunk = nn.Sequential(*trunk)
        if flatten:
            self.final_feat_dim = 64 * 19 * 19
        else:
            self.final_feat_dim = [64, 19, 19]

    def forward(self, x):
        out = self.trunk(x)
        return out

class ConvNetS(nn.Module):

    def __init__(self, depth, flatten=True):
        super(ConvNetS, self).__init__()
        trunk = []
        for i in range(depth):
            indim = 1 if i == 0 else 64
            outdim = 64
            B = ConvBlock(indim, outdim, pool=i < 4)
            trunk.append(B)
        if flatten:
            trunk.append(Flatten())
        self.trunk = nn.Sequential(*trunk)
        self.final_feat_dim = 64

    def forward(self, x):
        out = x[:, 0:1, :, :]
        out = self.trunk(out)
        return out

class ConvNetSNopool(nn.Module):

    def __init__(self, depth, flatten=False):
        super(ConvNetSNopool, self).__init__()
        trunk = []
        for i in range(depth):
            indim = 1 if i == 0 else 64
            outdim = 64
            B = ConvBlock(indim, outdim, pool=i in [0, 1], padding=0 if i in [0, 1] else 1)
            trunk.append(B)
        if flatten:
            trunk.append(Flatten())
        self.trunk = nn.Sequential(*trunk)
        if flatten:
            self.final_feat_dim = 64 * 19 * 19
        else:
            self.final_feat_dim = [64, 19, 19]

    def forward(self, x):
        out = x[:, 0:1, :, :]
        out = self.trunk(out)
        return out

class ResNetModel:

    def __init__(self, dataset, variant=34, flatten=False):
        super(ResNetModel, self).__init__()
        trunk = []
        dim = 4 if dataset == 'CIFAR' else 7
        self.final_feat_dim = 512 * dim * dim if flatten else [512, dim, dim]
        if variant == 18:
            resnet = models.resnet18(pretrained=True).to(device)
        elif variant == 34:
            resnet = models.resnet34(pretrained=True).to(device)
        self.model = nn.Sequential(*[*resnet.children()][:-2])

    def forward(self, x):
        out = self.model(x)
        return out

class ResNet(nn.Module):

    def __init__(self, block, layers, flatten=False):
        super(ResNet, self).__init__()
        dim = 7
        self.final_feat_dim = 512 * dim * dim if flatten else [512, dim, dim]
        self.initial_pool = False
        inplanes = self.inplanes = 64
        self.conv1 = nn.Conv2d(3, self.inplanes, kernel_size=5, stride=2, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(self.inplanes)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(block, inplanes, layers[0])
        self.layer2 = self._make_layer(block, inplanes * 2, layers[1], stride=2)
        self.layer3 = self._make_layer(block, inplanes * 4, layers[2], stride=2)
        self.layer4 = self._make_layer(block, inplanes * 8, layers[3], stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d(7)
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def _make_layer(self, block, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(conv1x1(self.inplanes, planes * block.expansion, stride), nn.BatchNorm2d(planes * block.expansion))
        layers = []
        layers.append(block(self.inplanes, planes, stride, downsample))
        self.inplanes = planes * block.expansion
        for _ in range(1, blocks):
            layers.append(block(self.inplanes, planes))
        return nn.Sequential(*layers)

    def forward(self, x, param_dict=None):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        if self.initial_pool:
            x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        return x

def Conv4(dataset, flatten=True):
    return ConvNet(4, dataset, flatten)

def Conv6(dataset, flatten=True):
    return ConvNet(6, dataset, flatten)

def Conv4NP(dataset, flatten=True):
    return ConvNetNopool(4, flatten)

def Conv6NP(dataset, flatten=True):
    return ConvNetNopool(6, flatten)

def Conv4S(dataset, flatten=True):
    return ConvNetS(4, flatten)

def Conv6S(dataset, flatten=True):
    return ConvNetS(6, flatten)

def Conv4SNP(dataset, flatten=True):
    return ConvNetSNopool(4, flatten)

def Conv6SNP(dataset, flatten=True):
    return ConvNetSNopool(6, flatten)

def ResNet12(FETI, dataset, flatten=True):
    if FETI:
        model = ResNet(BasicBlock, [2, 1, 1, 1], flatten)
        pretrained_dict = pretrain_load(pretrained_path)
        model.load_state_dict(pretrained_dict['state_dict'], strict=False)
    else:
        print('Torchvision.model does not support ResNet12. Change to ResNet18 instead.')
        model = ResNetModel(dataset, 18, flatten)
    return model

def ResNet18(FETI, dataset, flatten=True):
    if FETI:
        model = ResNet(BasicBlock, [2, 2, 2, 2], flatten)
        pretrained_dict = pretrain_load(pretrained_path)
        model.load_state_dict(pretrained_dict['state_dict'], strict=False)
    else:
        model = ResNetModel(dataset, 18, flatten)
    return model

def ResNet34(FETI, dataset, flatten=True):
    if FETI:
        model = ResNet(BasicBlock, [3, 4, 6, 3], flatten)
        pretrained_dict = pretrain_load(pretrained_path)
        model.load_state_dict(pretrained_dict['state_dict'], strict=False)
    else:
        model = ResNetModel(dataset, 34, flatten)
    return model


In [9]:
import numpy as np
import os
import glob
import datetime
model_dict = dict(Conv4=Conv4, Conv4S=Conv4S, Conv4NP=Conv4NP, Conv4SNP=Conv4SNP, Conv6=Conv6, Conv6S=Conv6S, Conv6NP=Conv6NP, Conv6SNP=Conv6SNP, ResNet12=ResNet12, ResNet18=ResNet18, ResNet34=ResNet34)

def get_assigned_file(checkpoint_dir, num):
    assign_file = os.path.join(checkpoint_dir, '{:d}.tar'.format(num))
    return assign_file

def get_best_file(checkpoint_dir):
    best_file = os.path.join(checkpoint_dir, 'best_model.tar')
    if os.path.isfile(best_file):
        return best_file


DATA HANDLING (Xử lý dữ liệu)

In [10]:
from . import datamgr
from . import dataset
from . import additional_transforms
from . import feature_loader


ImportError: attempted relative import with no known parent package

In [ ]:
import torch
from PIL import ImageEnhance
transformtypedict = dict(Brightness=ImageEnhance.Brightness, Contrast=ImageEnhance.Contrast, Sharpness=ImageEnhance.Sharpness, Color=ImageEnhance.Color)

class ImageJitter(object):

    def __init__(self, transformdict):
        self.transforms = [(transformtypedict[k], transformdict[k]) for k in transformdict]

    def __call__(self, img):
        out = img
        randtensor = torch.rand(len(self.transforms))
        for (i, (transformer, alpha)) in enumerate(self.transforms):
            r = alpha * (randtensor[i] * 2.0 - 1.0) + 1
            out = transformer(out).enhance(r).convert('RGB')
        return out


In [ ]:
import pdb
import torch
from PIL import Image
import json
import numpy as np
import random
import torchvision.transforms as transforms
import os
import cv2 as cv

def identity(x):
    """Identity function - returns input unchanged."""
    return x

class SetDataset:

    def __init__(self, data_file, batch_size, transform):
        with open(data_file, 'r') as f:
            self.meta = json.load(f)
        self.cl_list = np.unique(self.meta['image_labels']).tolist()
        self.sub_meta = {}
        for cl in self.cl_list:
            self.sub_meta[cl] = []
        for (x, y) in zip(self.meta['image_names'], self.meta['image_labels']):
            self.sub_meta[y].append(x)
        self.sub_dataloader = []
        sub_data_loader_params = dict(batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=False)
        for cl in self.cl_list:
            sub_dataset = SubDataset(self.sub_meta[cl], cl, transform=transform)
            self.sub_dataloader.append(torch.utils.data.DataLoader(sub_dataset, **sub_data_loader_params))

    def __getitem__(self, i):
        return next(iter(self.sub_dataloader[i]))

    def __len__(self):
        return len(self.cl_list)

class SubDataset:

    def __init__(self, sub_meta, cl, transform=transforms.ToTensor(), target_transform=identity):
        self.sub_meta = sub_meta
        self.cl = cl
        self.transform = transform
        self.target_transform = target_transform

    def __getitem__(self, i):
        seed = np.random.randint(2147483647)
        random.seed(seed)
        torch.manual_seed(seed)
        image_path = os.path.join(self.sub_meta[i])
        img = Image.open(image_path).convert('RGB')
        random.seed(seed)
        torch.manual_seed(seed)
        img = self.transform(img)
        target = self.target_transform(self.cl)
        return (img, target)

    def __len__(self):
        return len(self.sub_meta)

class EpisodicBatchSampler(object):

    def __init__(self, n_classes, n_way, n_episodes):
        self.n_classes = n_classes
        self.n_way = n_way
        self.n_episodes = n_episodes

    def __len__(self):
        return self.n_episodes

    def __iter__(self):
        for i in range(self.n_episodes):
            yield torch.randperm(self.n_classes)[:self.n_way]


In [ ]:
import pdb
import torch
from PIL import Image
import numpy as np
import random
import torchvision.transforms as transforms
import data.additional_transforms as add_transforms
from data.dataset import SetDataset, EpisodicBatchSampler
from abc import abstractmethod

class TransformLoader:

    def __init__(self, image_size, normalize_param=dict(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), jitter_param=dict(Brightness=0.4, Contrast=0.4, Color=0.4)):
        self.image_size = image_size
        self.normalize_param = normalize_param
        self.jitter_param = jitter_param

    def parse_transform(self, transform_type):
        if transform_type == 'ImageJitter':
            method = add_transforms.ImageJitter(self.jitter_param)
            return method
        method = getattr(transforms, transform_type)
        if transform_type == 'RandomSizedCrop':
            return method(self.image_size)
        elif transform_type == 'Resize':
            return method(self.image_size)
        elif transform_type == 'CenterCrop':
            return method(self.image_size)
        elif transform_type == 'Normalize':
            return method(**self.normalize_param)
        else:
            return method()

    def get_composed_transform(self, aug=False):
        if aug:
            transform_list = ['Resize', 'RandomSizedCrop', 'ColorJitter', 'RandomHorizontalFlip', 'ToTensor', 'Normalize']
        else:
            transform_list = ['Resize', 'CenterCrop', 'ToTensor', 'Normalize']
        transform_funcs = [self.parse_transform(x) for x in transform_list]
        transform = transforms.Compose(transform_funcs)
        return transform

class DataManager:

    @abstractmethod
    def get_data_loader(self, data_file, aug):
        pass

def seed_worker(worker_id):
    worker_seed = worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

class SetDataManager(DataManager):

    def __init__(self, image_size, n_way, k_shot, n_query, n_episode):
        super(SetDataManager, self).__init__()
        self.image_size = image_size
        self.n_way = n_way
        self.batch_size = k_shot + n_query
        self.n_episode = n_episode
        self.trans_loader = TransformLoader(image_size)

    def get_data_loader(self, data_file, aug):
        g = torch.Generator()
        g.manual_seed(0)
        transform = self.trans_loader.get_composed_transform(aug)
        dataset = SetDataset(data_file, self.batch_size, transform)
        sampler = EpisodicBatchSampler(len(dataset), self.n_way, self.n_episode)
        data_loader_params = dict(batch_sampler=sampler, num_workers=0, pin_memory=True, generator=g)
        data_loader = torch.utils.data.DataLoader(dataset, **data_loader_params)
        return data_loader


In [ ]:
import torch
import numpy as np
import h5py

class SimpleHDF5Dataset:

    def __init__(self, file_handle=None):
        if file_handle == None:
            self.f = ''
            self.all_feats_dset = []
            self.all_labels = []
            self.total = 0
        else:
            self.f = file_handle
            self.all_feats_dset = self.f['all_feats'][...]
            self.all_labels = self.f['all_labels'][...]
            self.total = self.f['count'][0]

    def __getitem__(self, i):
        return (torch.Tensor(self.all_feats_dset[i, :]), int(self.all_labels[i]))

    def __len__(self):
        return self.total

def init_loader(filename):
    with h5py.File(filename, 'r') as f:
        fileset = SimpleHDF5Dataset(f)
    feats = fileset.all_feats_dset
    labels = fileset.all_labels
    while np.sum(feats[-1]) == 0:
        feats = np.delete(feats, -1, axis=0)
        labels = np.delete(labels, -1, axis=0)
    class_list = np.unique(np.array(labels)).tolist()
    inds = range(len(labels))
    cl_data_file = {}
    for cl in class_list:
        cl_data_file[cl] = []
    for ind in inds:
        cl_data_file[labels[ind]].append(feats[ind])
    return cl_data_file


METHODS (Các phương pháp học)

In [ ]:
from . import meta_template
from . import transformer
from . import CTX


In [ ]:
import backbone
import torch
import torch.nn as nn
from torch.autograd import Variable
import numpy as np
import torch.nn.functional as F
import tqdm
from abc import abstractmethod
import pdb
import wandb
global device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
import warnings
warnings.filterwarnings('ignore')

class MetaTemplate(nn.Module):

    def __init__(self, model_func, n_way, k_shot, n_query, change_way=True):
        super(MetaTemplate, self).__init__()
        self.n_way = n_way
        self.k_shot = k_shot
        self.n_query = n_query
        self.feature = model_func()
        self.feat_dim = self.feature.final_feat_dim
        self.change_way = change_way

    @abstractmethod
    def set_forward(self, x, is_feature):
        pass

    @abstractmethod
    def set_forward_loss(self, x):
        pass

    def forward(self, x):
        out = self.feature.forward(x)
        return out

    def parse_feature(self, x, is_feature):
        x = Variable(x.to(device))
        if is_feature:
            z_all = x
        else:
            x = x.contiguous().view(self.n_way * (self.k_shot + self.n_query), *x.size()[2:])
            z_all = self.feature.forward(x)
            z_all = z_all.view(self.n_way, self.k_shot + self.n_query, *z_all.size()[1:])
        z_support = z_all[:, :self.k_shot]
        z_query = z_all[:, self.k_shot:]
        return (z_support, z_query)

    def correct(self, x):
        scores = self.set_forward(x)
        y_query = torch.from_numpy(np.repeat(range(self.n_way), self.n_query))
        y_query = Variable(y_query.to(device))
        (topk_scores, topk_labels) = scores.data.topk(1, 1, True, True)
        topk_ind = topk_labels
        top1_correct = (topk_ind[:, 0] == y_query).sum().item()
        return (float(top1_correct), len(y_query))

    def train_loop(self, epoch, num_epoch, train_loader, wandb_flag, optimizer):
        avg_loss = 0
        avg_acc = []
        with tqdm.tqdm(total=len(train_loader)) as train_pbar:
            for (i, (x, _)) in enumerate(train_loader):
                if self.change_way:
                    self.n_way = x.size(0)
                optimizer.zero_grad()
                (acc, loss) = self.set_forward_loss(x=x.to(device))
                loss.backward()
                optimizer.step()
                avg_loss += loss.item()
                avg_acc.append(acc)
                train_pbar.set_description('Epoch {:03d}/{:03d} | Acc {:.6f}  | Loss {:.6f}'.format(epoch + 1, num_epoch, np.mean(avg_acc) * 100, avg_loss / float(i + 1)))
                train_pbar.update(1)
        if wandb_flag:
            wandb.log({'Loss': avg_loss / float(i + 1), 'Train Acc': np.mean(avg_acc) * 100}, step=epoch + 1)

    def val_loop(self, val_loader, epoch, wandb_flag, record=None):
        correct = 0
        count = 0
        acc_all = []
        iter_num = len(val_loader)
        with tqdm.tqdm(total=len(val_loader)) as val_pbar:
            for (i, (x, _)) in enumerate(val_loader):
                if self.change_way:
                    self.n_way = x.size(0)
                (correct_this, count_this) = self.correct(x)
                acc_all.append(correct_this / count_this * 100)
                val_pbar.set_description('Validation    | Acc {:.6f}'.format(np.mean(acc_all)))
                val_pbar.update(1)
        acc_all = np.asarray(acc_all)
        acc_mean = np.mean(acc_all)
        acc_std = np.std(acc_all)
        if wandb_flag:
            wandb.log({'Val Acc': acc_mean}, step=epoch + 1)
        print('Val Acc = %4.2f%% +- %4.2f%%' % (acc_mean, 1.96 * acc_std / np.sqrt(iter_num)))
        return acc_mean


In [ ]:
import torch
import torch.nn as nn
from torch.autograd import Variable
import numpy as np
import torch.nn.functional as F
from abc import abstractmethod
from methods.meta_template import MetaTemplate
from einops import rearrange, repeat
from einops.layers.torch import Rearrange
from backbone import CosineDistLinear
import pdb
import IPython
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

class FewShotTransformer(MetaTemplate):

    def __init__(self, model_func, n_way, k_shot, n_query, variant='softmax', depth=1, heads=8, dim_head=64, mlp_dim=512):
        super(FewShotTransformer, self).__init__(model_func, n_way, k_shot, n_query)
        self.loss_fn = nn.CrossEntropyLoss()
        self.k_shot = k_shot
        self.variant = variant
        self.depth = depth
        dim = self.feat_dim
        self.ATTN = Attention(dim, heads=heads, dim_head=dim_head, variant=variant)
        self.sm = nn.Softmax(dim=-2)
        self.proto_weight = nn.Parameter(torch.ones(n_way, k_shot, 1))
        self.FFN = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, mlp_dim), nn.GELU(), nn.Linear(mlp_dim, dim))
        self.linear = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, dim_head), CosineDistLinear(dim_head, 1) if variant == 'cosine' else nn.Linear(dim_head, 1))

    def set_forward(self, x, is_feature=False):
        (z_support, z_query) = self.parse_feature(x, is_feature)
        z_support = z_support.contiguous().view(self.n_way, self.k_shot, -1)
        z_proto = (z_support * self.sm(self.proto_weight)).sum(1).unsqueeze(0)
        z_query = z_query.contiguous().view(self.n_way * self.n_query, -1).unsqueeze(1)
        (x, query) = (z_proto, z_query)
        for _ in range(self.depth):
            x = self.ATTN(q=x, k=query, v=query) + x
            x = self.FFN(x) + x
        return self.linear(x).squeeze()

    def set_forward_loss(self, x):
        target = torch.from_numpy(np.repeat(range(self.n_way), self.n_query))
        target = Variable(target.to(device))
        scores = self.set_forward(x)
        loss = self.loss_fn(scores, target)
        predict = torch.argmax(scores, dim=1)
        acc = (predict == target).sum().item() / target.size(0)
        return (acc, loss)

class Attention(nn.Module):

    def __init__(self, dim, heads, dim_head, variant):
        super().__init__()
        inner_dim = heads * dim_head
        project_out = not (heads == 1 and dim_head == dim)
        self.heads = heads
        self.scale = dim_head ** (-0.5)
        self.sm = nn.Softmax(dim=-1)
        self.variant = variant
        self.input_linear = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, inner_dim, bias=False))
        self.output_linear = nn.Linear(inner_dim, dim) if project_out else nn.Identity()

    def forward(self, q, k, v):
        (f_q, f_k, f_v) = map(lambda t: rearrange(self.input_linear(t), 'q n (h d) ->  h q n d', h=self.heads), (q, k, v))
        if self.variant == 'cosine':
            dots = cosine_distance(f_q, f_k.transpose(-1, -2))
            out = torch.matmul(dots, f_v)
        else:
            dots = torch.matmul(f_q, f_k.transpose(-1, -2)) * self.scale
            out = torch.matmul(self.sm(dots), f_v)
        out = rearrange(out, 'h q n d -> q n (h d)')
        return self.output_linear(out)

def cosine_distance(x1, x2):
    """
    x1      =  [b, h, n, k]
    x2      =  [b, h, k, m]
    output  =  [b, h, n, m]
    """
    dots = torch.matmul(x1, x2)
    scale = torch.einsum('bhi, bhj -> bhij', (torch.norm(x1, 2, dim=-1), torch.norm(x2, 2, dim=-2)))
    return dots / scale


In [ ]:
"""
Paper: "CrossTransformers: spatially-aware few-shot transfer"
Arxiv: https://arxiv.org/abs/2007.11498
This code is modified from https://github.com/lucidrains/cross-transformers-pytorch
"""
import torch
import torch.nn as nn
from torch.autograd import Variable
import numpy as np
import torch.nn.functional as F
from abc import abstractmethod
from methods.meta_template import MetaTemplate
from einops import rearrange, repeat
from einops.layers.torch import Rearrange
from backbone import CosineDistLinear
import pdb
import IPython
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

class CTX(MetaTemplate):

    def __init__(self, model_func, n_way, k_shot, n_query, heatmap=0, variant='softmax', input_dim=64, dim_attn=128):
        super(CTX, self).__init__(model_func, n_way, k_shot, n_query)
        self.loss_fn = nn.CrossEntropyLoss()
        self.n_way = n_way
        self.k_shot = k_shot
        self.attn = variant
        self.sm = nn.Softmax(dim=-1)
        self.dim_attn = dim_attn
        self.linear_attn = nn.Conv2d(input_dim, dim_attn, 1, bias=False)

    def set_forward(self, x, is_feature=False):
        """
        dimensions names:

        b - batch
        n - n way
        k - k shot
        q - num query
        c, h, w - img shape
        d - dim
        """
        (z_support, z_query) = self.parse_feature(x, is_feature)
        (z_query, z_support) = map(lambda t: rearrange(t, 'b n c h w -> (b n) c h w'), (z_query, z_support))
        (query_q, query_v, support_k, support_v) = map(lambda t: self.linear_attn(t), (z_query, z_query, z_support, z_support))
        (query_q, query_v) = map(lambda t: rearrange(t, 'q c h w -> q () (c h w)'), (query_q, query_v))
        (support_k, support_v) = map(lambda t: rearrange(t, '(n k) c h w -> n (c h w) k', n=self.n_way, k=self.k_shot), (support_k, support_v))
        query_q = rearrange(query_q, 'q b d -> b q d')
        if self.attn == 'softmax':
            dots = torch.matmul(query_q, support_k)
            scale = self.dim_attn ** 0.5
            attn_weights = self.sm(dots / scale)
        else:
            dots = torch.matmul(query_q, support_k)
            scale = torch.einsum('bq, nk -> nqk', (torch.norm(query_q, 2, dim=-1), torch.norm(support_k, 2, dim=-2)))
            attn_weights = dots / scale
        out = torch.einsum('nqk, ndk -> qnd', attn_weights, support_v)
        euclidean_dist = -((query_v - out) ** 2).sum(dim=-1) / (self.feat_dim[1] * self.feat_dim[2])
        return euclidean_dist

    def set_forward_loss(self, x):
        target = torch.from_numpy(np.repeat(range(self.n_way), self.n_query))
        target = Variable(target.to(device))
        scores = self.set_forward(x)
        loss = self.loss_fn(scores, target)
        predict = torch.argmax(scores, dim=1)
        acc = (predict == target).sum().item() / target.size(0)
        return (acc, loss)


MAIN TRAINING/TESTING

In [ ]:
import glob
import json
import os
import pdb
import pprint
import random
import time
import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.optim
import torch.optim.lr_scheduler as lr_scheduler
import torch.utils.data.sampler
import tqdm
from torch.autograd import Variable
from torchsummary import summary
import backbone
import configs
import data.feature_loader as feat_loader
import wandb
from data.datamgr import SetDataManager
from io_utils import get_assigned_file, get_best_file, model_dict, parse_args
from methods.CTX import CTX
from methods.transformer import FewShotTransformer
global device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def train(base_loader, val_loader, model, optimization, num_epoch, params):
    if optimization == 'Adam':
        optimizer = torch.optim.Adam(model.parameters(), lr=params.learning_rate, weight_decay=params.weight_decay)
    elif optimization == 'AdamW':
        optimizer = torch.optim.AdamW(model.parameters(), lr=params.learning_rate, weight_decay=params.weight_decay)
    elif optimization == 'SGD':
        optimizer = torch.optim.SGD(model.parameters(), lr=params.learning_rate, momentum=params.momentum, weight_decay=params.weight_decay)
    else:
        raise ValueError('Unknown optimization, please define by yourself')
    max_acc = 0
    for epoch in range(num_epoch):
        model.train()
        model.train_loop(epoch, num_epoch, base_loader, params.wandb, optimizer)
        with torch.no_grad():
            model.eval()
            if not os.path.isdir(params.checkpoint_dir):
                os.makedirs(params.checkpoint_dir)
            acc = model.val_loop(val_loader, epoch, params.wandb)
            if acc > max_acc:
                print('best model! save...')
                max_acc = acc
                outfile = os.path.join(params.checkpoint_dir, 'best_model.tar')
                torch.save({'epoch': epoch, 'state': model.state_dict()}, outfile)
            if epoch % params.save_freq == 0 or epoch == num_epoch - 1:
                outfile = os.path.join(params.checkpoint_dir, '{:d}.tar'.format(epoch))
                torch.save({'epoch': epoch, 'state': model.state_dict()}, outfile)
        print()
    return model

def direct_test(test_loader, model, params):
    correct = 0
    count = 0
    acc = []
    iter_num = len(test_loader)
    with tqdm.tqdm(total=len(test_loader)) as pbar:
        for (i, (x, _)) in enumerate(test_loader):
            scores = model.set_forward(x)
            pred = scores.data.cpu().numpy().argmax(axis=1)
            y = np.repeat(range(params.n_way), pred.shape[0] // params.n_way)
            acc.append(np.mean(pred == y) * 100)
            pbar.set_description('Test       | Acc {:.6f}'.format(np.mean(acc)))
            pbar.update(1)
    acc_all = np.asarray(acc)
    acc_mean = np.mean(acc_all)
    acc_std = np.std(acc_all)
    return (acc_mean, acc_std)

def seed_func():
    seed = 4040
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(10)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def change_model(model_name):
    if model_name == 'Conv4':
        model_name = 'Conv4NP'
    elif model_name == 'Conv6':
        model_name = 'Conv6NP'
    elif model_name == 'Conv4S':
        model_name = 'Conv4SNP'
    elif model_name == 'Conv6S':
        model_name = 'Conv6SNP'
    return model_name
if __name__ == '__main__':
    params = parse_args()
    pp = pprint.PrettyPrinter(indent=4)
    pp.pprint(vars(params))
    print()
    project_name = 'Few-Shot_TransFormer'
    if params.dataset == 'Omniglot':
        params.n_query = 15
    if params.wandb:
        wandb_name = params.method + '_' + params.backbone + '_' + params.dataset + '_' + str(params.n_way) + 'w' + str(params.k_shot) + 's'
        if params.train_aug:
            wandb_name += '_aug'
        if params.FETI and 'ResNet' in params.backbone:
            wandb_name += '_FETI'
        wandb_name += '_' + params.datetime
        wandb.init(project=project_name, name=wandb_name, config=params, id=params.datetime)
        print()
    if params.dataset == 'cross':
        base_file = configs.data_dir['miniImagenet'] + 'all.json'
        val_file = configs.data_dir['CUB'] + 'val.json'
    elif params.dataset == 'cross_char':
        base_file = configs.data_dir['Omniglot'] + 'noLatin.json'
        val_file = configs.data_dir['emnist'] + 'val.json'
    else:
        base_file = configs.data_dir[params.dataset] + 'base.json'
        val_file = configs.data_dir[params.dataset] + 'val.json'
    if params.dataset == 'CIFAR':
        image_size = 112 if 'ResNet' in params.backbone else 64
    else:
        image_size = 224 if 'ResNet' in params.backbone else 84
    if params.dataset in ['Omniglot', 'cross_char']:
        if params.backbone == 'Conv4':
            params.backbone = 'Conv4S'
        if params.backbone == 'Conv6':
            params.backbone = 'Conv6S'
    optimization = params.optimization
    if params.method in ['FSCT_softmax', 'FSCT_cosine', 'CTX_softmax', 'CTX_cosine']:
        few_shot_params = dict(n_way=params.n_way, k_shot=params.k_shot, n_query=params.n_query)
        base_datamgr = SetDataManager(image_size, n_episode=params.n_episode, **few_shot_params)
        base_loader = base_datamgr.get_data_loader(base_file, aug=params.train_aug)
        val_datamgr = SetDataManager(image_size, n_episode=params.n_episode, **few_shot_params)
        val_loader = val_datamgr.get_data_loader(val_file, aug=False)
        seed_func()
        if params.method in ['FSCT_softmax', 'FSCT_cosine']:
            variant = 'cosine' if params.method == 'FSCT_cosine' else 'softmax'

            def feature_model():
                if params.dataset in ['Omniglot', 'cross_char']:
                    params.backbone = change_model(params.backbone)
                return model_dict[params.backbone](params.FETI, params.dataset, flatten=True) if 'ResNet' in params.backbone else model_dict[params.backbone](params.dataset, flatten=True)
            model = FewShotTransformer(feature_model, variant=variant, **few_shot_params)
        elif params.method in ['CTX_softmax', 'CTX_cosine']:
            variant = 'cosine' if params.method == 'CTX_cosine' else 'softmax'
            input_dim = 512 if 'ResNet' in params.backbone else 64

            def feature_model():
                if params.dataset in ['Omniglot', 'cross_char']:
                    params.backbone = change_model(params.backbone)
                return model_dict[params.backbone](params.FETI, params.dataset, flatten=False) if 'ResNet' in params.backbone else model_dict[params.backbone](params.dataset, flatten=False)
            model = CTX(feature_model, variant=variant, input_dim=input_dim, **few_shot_params)
    else:
        raise ValueError('Unknown method')
    model = model.to(device)
    params.checkpoint_dir = '%scheckpoints/%s/%s_%s' % (configs.save_dir, params.dataset, params.backbone, params.method)
    if params.train_aug:
        params.checkpoint_dir += '_aug'
    if params.FETI and 'ResNet' in params.backbone:
        params.checkpoint_dir += '_FETI'
    params.checkpoint_dir += '_%dway_%dshot' % (params.n_way, params.k_shot)
    if not os.path.isdir(params.checkpoint_dir):
        os.makedirs(params.checkpoint_dir)
    print('===================================')
    print('Train phase: ')
    model = train(base_loader, val_loader, model, optimization, params.num_epoch, params)
    print('===================================')
    print('Test phase: ')
    iter_num = params.test_iter
    split = params.split
    if params.dataset == 'cross':
        if split == 'base':
            testfile = configs.data_dir['miniImagenet'] + 'all.json'
        else:
            testfile = configs.data_dir['CUB'] + split + '.json'
    elif params.dataset == 'cross_char':
        if split == 'base':
            testfile = configs.data_dir['Omniglot'] + 'noLatin.json'
        else:
            testfile = configs.data_dir['emnist'] + split + '.json'
    else:
        testfile = configs.data_dir[params.dataset] + split + '.json'
    if params.save_iter != -1:
        modelfile = get_assigned_file(params.checkpoint_dir, params.save_iter)
    else:
        modelfile = get_best_file(params.checkpoint_dir)
    test_datamgr = SetDataManager(image_size, n_episode=iter_num, **few_shot_params)
    test_loader = test_datamgr.get_data_loader(testfile, aug=False)
    acc_all = []
    model = model.to(device)
    root = os.getcwd()
    if params.save_iter != -1:
        modelfile = get_assigned_file(params.checkpoint_dir, params.save_iter)
    else:
        modelfile = get_best_file(params.checkpoint_dir)
    if modelfile is not None:
        tmp = torch.load(modelfile)
        model.load_state_dict(tmp['state'])
    split = params.split
    if params.save_iter != -1:
        split_str = split + '_' + str(params.save_iter)
    else:
        split_str = split
    (acc_mean, acc_std) = direct_test(test_loader, model, params)
    print('%d Test Acc = %4.2f%% +- %4.2f%%' % (iter_num, acc_mean, 1.96 * acc_std / np.sqrt(iter_num)))
    if params.wandb:
        wandb.log({'Test Acc': acc_mean})
    with open('./record/results.txt', 'a') as f:
        timestamp = params.datetime
        aug_str = '-aug' if params.train_aug else ''
        aug_str += '-FETI' if params.FETI and 'ResNet' in params.backbone else ''
        if params.backbone == 'Conv4SNP':
            params.backbone = 'Conv4'
        elif params.backbone == 'Conv6SNP':
            params.backbone = 'Conv6'
        exp_setting = '%s-%s-%s%s-%sw%ss' % (params.dataset, params.backbone, params.method, aug_str, params.n_way, params.k_shot)
        acc_str = 'Test Acc = %4.2f%% +- %4.2f%%' % (acc_mean, 1.96 * acc_std / np.sqrt(iter_num))
        f.write('Time: %s   Setting: %s %s \n' % (timestamp, exp_setting.ljust(50), acc_str))
    wandb.finish()


In [ ]:
import glob
import json
import os
import pdb
import pprint
import random
import time
import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.optim
import torch.optim.lr_scheduler as lr_scheduler
import torch.utils.data.sampler
import tqdm
from torch.autograd import Variable
from torchsummary import summary
import backbone
import configs
import data.feature_loader as feat_loader
import wandb
from data.datamgr import SetDataManager
from io_utils import get_assigned_file, get_best_file, model_dict, parse_args
from methods.CTX import CTX
from methods.transformer import FewShotTransformer
global device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def direct_test(test_loader, model, params):
    correct = 0
    count = 0
    acc = []
    iter_num = len(test_loader)
    with tqdm.tqdm(total=len(test_loader)) as pbar:
        for (i, (x, _)) in enumerate(test_loader):
            scores = model.set_forward(x)
            pred = scores.data.cpu().numpy().argmax(axis=1)
            y = np.repeat(range(params.n_way), pred.shape[0] // params.n_way)
            acc.append(np.mean(pred == y) * 100)
            pbar.set_description('Test       | Acc {:.6f}'.format(np.mean(acc)))
            pbar.update(1)
    acc_all = np.asarray(acc)
    acc_mean = np.mean(acc_all)
    acc_std = np.std(acc_all)
    return (acc_mean, acc_std)

def seed_func():
    seed = 4040
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(10)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def change_model(model_name):
    if model_name == 'Conv4':
        model_name = 'Conv4NP'
    elif model_name == 'Conv6':
        model_name = 'Conv6NP'
    elif model_name == 'Conv4S':
        model_name = 'Conv4SNP'
    elif model_name == 'Conv6S':
        model_name = 'Conv6SNP'
    return model_name
if __name__ == '__main__':
    params = parse_args()
    pp = pprint.PrettyPrinter(indent=4)
    pp.pprint(vars(params))
    print()
    if params.dataset == 'Omniglot':
        params.n_query = min(params.n_query, 15)
    if params.dataset == 'CIFAR':
        image_size = 112 if 'ResNet' in params.backbone else 64
    else:
        image_size = 224 if 'ResNet' in params.backbone else 84
    if params.dataset in ['Omniglot', 'cross_char']:
        if params.backbone == 'Conv4':
            params.backbone = 'Conv4S'
        if params.backbone == 'Conv6':
            params.backbone = 'Conv6S'
    iter_num = params.test_iter
    split = params.split
    if params.dataset == 'cross':
        if split == 'base':
            testfile = configs.data_dir['miniImagenet'] + 'all.json'
        else:
            testfile = configs.data_dir['CUB'] + split + '.json'
    elif params.dataset == 'cross_char':
        if split == 'base':
            testfile = configs.data_dir['Omniglot'] + 'noLatin.json'
        else:
            testfile = configs.data_dir['emnist'] + split + '.json'
    else:
        testfile = configs.data_dir[params.dataset] + split + '.json'
    if params.method in ['FSCT_softmax', 'FSCT_cosine', 'CTX_softmax', 'CTX_cosine']:
        seed_func()
        few_shot_params = dict(n_way=params.n_way, k_shot=params.k_shot, n_query=params.n_query)
        if params.method in ['FSCT_softmax', 'FSCT_cosine']:
            variant = 'cosine' if params.method == 'FSCT_cosine' else 'softmax'

            def feature_model():
                if params.dataset in ['Omniglot', 'cross_char']:
                    params.backbone = change_model(params.backbone)
                return model_dict[params.backbone](params.FETI, params.dataset, flatten=True) if 'ResNet' in params.backbone else model_dict[params.backbone](params.dataset, flatten=True)
            model = FewShotTransformer(feature_model, variant=variant, **few_shot_params)
        elif params.method in ['CTX_softmax', 'CTX_cosine']:
            variant = 'cosine' if params.method == 'CTX_cosine' else 'softmax'
            input_dim = 512 if 'ResNet' in params.backbone else 64

            def feature_model():
                if params.dataset in ['Omniglot', 'cross_char']:
                    params.backbone = change_model(params.backbone)
                return model_dict[params.backbone](params.FETI, params.dataset, flatten=False) if 'ResNet' in params.backbone else model_dict[params.backbone](params.dataset, flatten=False)
            model = CTX(feature_model, variant=variant, input_dim=input_dim, **few_shot_params)
    else:
        raise ValueError('Unknown method')
    model = model.to(device)
    params.checkpoint_dir = '%scheckpoints/%s/%s_%s' % (configs.save_dir, params.dataset, params.backbone, params.method)
    if params.train_aug:
        params.checkpoint_dir += '_aug'
    if params.FETI and 'ResNet' in params.backbone:
        params.checkpoint_dir += '_FETI'
    params.checkpoint_dir += '_%dway_%dshot' % (params.n_way, params.k_shot)
    if not os.path.isdir(params.checkpoint_dir):
        raise ValueError("Can't find save model dir")
    print('===================================')
    print('Test phase: ')
    if params.save_iter != -1:
        modelfile = get_assigned_file(params.checkpoint_dir, params.save_iter)
    else:
        modelfile = get_best_file(params.checkpoint_dir)
    test_datamgr = SetDataManager(image_size, n_episode=iter_num, **few_shot_params)
    test_loader = test_datamgr.get_data_loader(testfile, aug=False)
    acc_all = []
    model = model.to(device)
    root = os.getcwd()
    if params.save_iter != -1:
        modelfile = get_assigned_file(params.checkpoint_dir, params.save_iter)
    else:
        modelfile = get_best_file(params.checkpoint_dir)
    if modelfile is not None:
        tmp = torch.load(modelfile)
        model.load_state_dict(tmp['state'])
    split = params.split
    if params.save_iter != -1:
        split_str = split + '_' + str(params.save_iter)
    else:
        split_str = split
    (acc_mean, acc_std) = direct_test(test_loader, model, params)
    print('%d Test Acc = %4.2f%% +- %4.2f%%' % (iter_num, acc_mean, 1.96 * acc_std / np.sqrt(iter_num)))
    with open('./record/results.txt', 'a') as f:
        timestamp = params.datetime
        aug_str = '-aug' if params.train_aug else ''
        aug_str += '-FETI' if params.FETI and 'ResNet' in params.backbone else ''
        if params.backbone == 'Conv4SNP':
            params.backbone = 'Conv4'
        elif params.backbone == 'Conv6SNP':
            params.backbone = 'Conv6'
        exp_setting = '%s-%s-%s%s-%sw%ss' % (params.dataset, params.backbone, params.method, aug_str, params.n_way, params.k_shot)
        acc_str = 'Test Acc = %4.2f%% +- %4.2f%%' % (acc_mean, 1.96 * acc_std / np.sqrt(iter_num))
        f.write('Time: %s   Setting: %s %s \n' % (timestamp, exp_setting.ljust(50), acc_str))


UTILITIES & VISUALIZATION